In [12]:
FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'

In [23]:
from abc import ABC, abstractmethod
from common import *
import plotly

# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        return [sum(data[:i+1])/(i+1) for i in range(len(data))]

# 데이터 시각화 전략 인터페이스
class VisualizationStrategy(ABC):
    @abstractmethod
    def visualize(self, data):
        pass

# 구체적인 시각화 전략: 라인 그래프
class LineGraphVisualization(VisualizationStrategy):
    def visualize(self, data):
        print("데이터를 그래프로 시각화합니다.")
        pass

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
                 visualizer: VisualizationStrategy, notifier: NotificationStrategy):
        self.data_loader = data_loader
        self.processor = processor
        self.visualizer = visualizer
        self.notifier = notifier
        self.data = None
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [24]:
bitmex = Bitmex(BitmexCSVDataLoader(15,'2023-12-31 15:01:00'), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
CSV 데이터 로드 완료.
입력받은 15분봉으로 2023-12-31 15:01:00 까지 표현합니다.
전처리 완료


In [10]:
bitmex.data

,timestamp,symbol,open,high,low,close,trades,volume,vwap,lastSize,turnover,homeNotional,foreignNotional
0,2015-09-25 12:01:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
1,2015-09-25 12:02:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
2,2015-09-25 12:03:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
3,2015-09-25 12:04:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
4,2015-09-25 12:05:00+00:00,XBTUSD,NaN,NaN,NaN,NaN,0,0,NaN,NaN,0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4005552,2023-05-08 03:13:00+00:00,XBTUSD,28320.5,28328.5,28320.5,28328.5,12,50000,28322.75,200.0,176536630,1.765366,50000.0
4005553,2023-05-08 03:14:00+00:00,XBTUSD,28328.5,28330.0,28329.5,28330.0,11,43500,28329.65,200.0,153549633,1.535496,43500.0
4005554,2023-05-08 03:15:00+00:00,XBTUSD,28330.0,28329.5,28320.0,28320.0,51,173600,28326.10,100.0,612862282,6.128623,173600.0
4005555,2023-05-08 03:16:00+00:00,XBTUSD,28320.0,28322.5,28289.5,28304.0,154,1151700,28313.05,3300.0,4067744000,40.677440,1151700.0
